# Week 19: Time-Series Deep Learning, State Space Models & Mamba
## Notebook 04: Structured State Space Models (S4) Foundations from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In sequence modeling, researchers faced an agonizing trade-off:
- **Transformers:** Quadratic attention $O(N^2)$ memory and compute. Fast parallel training, but slow autoregressive inference.
- **RNNs:** Linear $O(N)$ inference and constant memory. But slow sequential training and cannot maintain memory over 10,000+ steps.

In 2021, Albert Gu (Stanford) introduced **S4 (Structured State Space Models)**:
- Roots in classical control theory and continuous-time differential equations.
- **Dual Representation:** Trains in parallel as a **Convolution** ($O(N \log N)$ via FFT), but generates tokens autoregressively as an **RNN** ($O(1)$ per step)!
- Maintains memory over **100,000+ tokens**!

---

### 2. Mathematical Foundations: Continuous-to-Discrete ODE
Continuous Linear Time-Invariant (LTI) State Space equation:
$$h'(t) = A h(t) + B x(t)$$
$$y(t) = C h(t) + D x(t)$$

Where:
- $h(t) \in \mathbb{R}^N$: Latent state vector.
- $A \in \mathbb{R}^{N \times N}$: State transition matrix.
- $B, C, D$: Input, output, and skip projections.

#### Bilinear (Tustin) Discretization with step size $\Delta$:
$$\bar{A} = \left( I - \frac{\Delta}{2} A \right)^{-1} \left( I + \frac{\Delta}{2} A \right)$$
$$\bar{B} = \left( I - \frac{\Delta}{2} A \right)^{-1} \Delta B$$

Discrete Recurrent View:
$$h_t = \bar{A} h_{t-1} + \bar{B} x_t, \quad y_t = C h_t$$

Discrete Convolutional View (Global Filter Kernel $\bar{K}$):
$$\bar{K} = \left( C \bar{B}, \; C \bar{A} \bar{B}, \; C \bar{A}^2 \bar{B}, \; \dots, \; C \bar{A}^{L-1} \bar{B} \right)$$
$$y = x * \bar{K}$$

In [ ]:
import numpy as np

# Step 1: Define Continuous 1D State Space System (N=4 latent state dimension)
N = 4
delta = 0.1  # Discretization step size

# HiPPO-inspired continuous transition matrix A
# (Negative diagonal ensures stability)
A = -np.eye(N) * 0.5
for i in range(N):
    for j in range(i):
        A[i, j] = -1.0

B = np.ones((N, 1))
C = np.array([[1.0, -0.5, 0.25, -0.1]])

print("Continuous Matrix A (4x4):\n", A)

# Step 2: Discretize using Bilinear (Tustin) Transform
# A_bar = inv(I - delta/2 * A) @ (I + delta/2 * A)
I = np.eye(N)
inv_term = np.linalg.inv(I - (delta / 2.0) * A)
A_bar = inv_term @ (I + (delta / 2.0) * A)
B_bar = (inv_term * delta) @ B

print("\nDiscretized State Transition Matrix A_bar:\n", np.round(A_bar, 3))

# Step 3: Dual Representation Equivalence Verification
# Input sequence of 6 time steps
x_seq = np.array([1.0, 0.5, -0.5, 2.0, 1.0, -1.0])
L = len(x_seq)

# Representation 1: Recurrent Step-by-Step (Linear Time O(L), O(1) memory)
h = np.zeros((N, 1))
y_recurrent = []
for t in range(L):
    h = A_bar @ h + B_bar * x_seq[t]
    out = C @ h
    y_recurrent.append(float(out[0, 0]))
y_recurrent = np.array(y_recurrent)

# Representation 2: Convolutional Global Kernel K_bar = [C B_bar, C A_bar B_bar, ...]
K_bar = []
A_power = np.eye(N)
for t in range(L):
    kernel_t = C @ A_power @ B_bar
    K_bar.append(float(kernel_t[0, 0]))
    A_power = A_power @ A_bar
K_bar = np.array(K_bar)

# Convolve input with K_bar (Full parallel training view!)
y_conv = np.convolve(x_seq, K_bar)[:L]

print("\n--- Dual Representation Output Verification ---")
print("Recurrent Output (Sequential Inference):\n", np.round(y_recurrent, 4))
print("Convolutional Output (Parallel Training):\n", np.round(y_conv, 4))
print("Exact Match:", np.allclose(y_recurrent, y_conv))

### 3. Step-by-Step Code Walkthrough
1. **Bilinear Discretization:** Maps continuous differential operators into discrete transition matrices $(\bar{A}, \bar{B})$.
2. **Recurrent Inference:** $h_t = \bar{A} h_{t-1} + \bar{B} x_t$ processes each token in $O(1)$ constant memory and latency.
3. **Convolutional Equivalence:** Precomputing the convolution kernel $\bar{K}$ and executing 1D convolution yields **exact numerical equivalence**, allowing fast parallel GPU training without sequential bottlenecks!

---

### 4. Industry Applications & Trade-offs
- **Long Sequence Audio & DNA Modeling:** S4 solved the Long Range Arena (LRA) benchmark across 16,000-token sequences where Transformers failed or ran out of VRAM.
- **Limitation:** Linear Time-Invariant (LTI) systems have static parameters ($A, B, C$) that cannot dynamically adapt to input content (leading to the invention of **Mamba**!).

---

### 5. Top Interview Questions & Answers
**Q1: What is the HiPPO matrix and why is it essential in S4?**
> **Answer:** Standard random initialization of $A$ causes memory to vanish over long contexts. HiPPO (High-order Polynomial Projection Operators) mathematically derives an optimal matrix $A$ that maintains continuous polynomial memory over past history with respect to a sliding measure, preserving long-range dependencies.

**Q2: Why can S4 be computed as a convolution during training?**
> **Answer:** Because $A, B, C$ are time-invariant (constant for all time steps). Unrolling the recurrence yields $y_t = \sum_{k=0}^t (C \bar{A}^k \bar{B}) x_{t-k}$, which is mathematically identical to a 1D convolution with kernel $K_k = C \bar{A}^k \bar{B}$.